# 2. Classify unseen profiles

Applies the published model to campaign profiles that have no labels. The
example uses PS111; point `input_path` at any file with the same features.

In [ ]:
import sys
sys.path.append("../src")

import pickle
import numpy as np
import torch
import xarray as xr
import matplotlib.pyplot as plt
from torch import nn
from torch.utils.data import DataLoader

import config
import data
import evaluation
import inference
import model
import plotting
import training

## Load the model and the constants it was trained with

Recomputing the constants from the new campaign would rescale the inputs away
from what the model learned, and the predictions would quietly get worse rather
than fail outright.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

net = model.CNN1D()
net.load_state_dict(torch.load(config.CHECKPOINT, map_location=device))
net.eval().to(device)

norm = pickle.load(open(config.NORM_CONSTANTS, "rb"))
print(f"model on {device}")

## Load the profiles to classify

In [ ]:
input_path = config.UNLABELLED_DATA
unlabelled = xr.open_dataset(input_path)

missing = [v for v in config.FEATURE_VARS if v not in unlabelled.data_vars]
if missing:
    raise ValueError(f"input is missing required features: {missing}")

print(f"profiles {unlabelled.sizes['profile']}, depth bins {unlabelled.sizes['depth_bins']}")

Have a look at one profile before classifying it.

In [ ]:
profile_index = 0
profile = unlabelled.isel(profile=profile_index)

plt.figure(figsize=(10, 7))
for var in config.FEATURE_VARS:
    plt.plot(profile[var], unlabelled.depth_bins, label=var)

plt.gca().invert_yaxis()
plt.xlabel("value")
plt.ylabel("depth bin")
plt.title(f"Profile {profile_index}: all input features")
plt.legend(loc="center left", bbox_to_anchor=(1, 0.5), fontsize=8)
plt.tight_layout()
plt.show()

## Run inference

Profiles with no valid depth bins are skipped and reported, rather than
silently shifting the results.

In [ ]:
profile_ids = np.unique(unlabelled.profile.values)
prediction_set = data.SMPProfileDataset(unlabelled, profile_ids, norm,
                                        include_labels=False)

print(f"classifying {len(prediction_set)} of {len(profile_ids)} profiles")
if prediction_set.skipped_profile_ids:
    print(f"skipped: {prediction_set.skipped_profile_ids}")

predictions = inference.predict_profiles(net, prediction_set, device)

predicted = np.concatenate(predictions)
for i, name in enumerate(config.CLASS_NAMES):
    count = int((predicted == i).sum())
    print(f"{i}  {name:<20} {count:>9,} bins  ({count / len(predicted):5.1%})")

## Inspect a classified profile

In [ ]:
features, _ = prediction_set[profile_index]
plotting.plot_profile_predictions(features, predictions[profile_index])
plt.show()

## Write the predictions back to NetCDF

In [ ]:
unlabelled = inference.add_predictions_to_dataset(
    unlabelled, predictions, prediction_set.profile_ids)

output_path = config.REPO_ROOT / "outputs" / f"{input_path.stem}_with_predictions.nc"
output_path.parent.mkdir(parents=True, exist_ok=True)
unlabelled.to_netcdf(output_path, mode="w", format="NETCDF4", engine="netcdf4")

print(f"wrote {output_path}")
unlabelled.predicted_labels